# Tutorial 0: Data Preparation

This tutorial explains how to prepare your data in a format that the `catella` package can handle. The aim here is to create a table of methylation probabilities for individual bases of each sequence from the `bam` file. `catella` uses this table to predict nucleosome positions. We will specifically discuss two main cases for preparing the data:

* Artificial constructs - data generated from synthetic sequences (e.g., repeats of the 601 widom sequence), where all molecules from the same sequence have the same size
* Native fragments - data generated from in-vivo systems, where molecules of various lengths are extracted by restriction enzymes

## Software prerequisites

The following software packages are required for this tutorial:

* `modkit` - We use this package to extract the methylation signal. It can be installed via `conda`, and its documentation is available here - https://nanoporetech.github.io/modkit. The code below has been tested using `modkit` version 0.6.3.
* `samtools` - We use this tool to index the genome `fasta` file and generate a file storing sequence names and sizes.
* `bedtools` - We use this tool to perform intersection between `bed` files. This is only needed for handling native fragments.

## Upstream preprocesing assumptions

The steps below assume that the sequencing data have been preprocessed as follows:

* There is fasta file containing the reference genome of the chromosomes or sequences of interest (e.g., `genome.fa`).
* Base calling has been done (e.g., using `pbmm2` for PacBio data or `minimap2` for Oxford Nanopore data), resulting in an aligned `bam` file (e.g., `input.bam`).

<div class="alert alert-info">
    
**Note:** The steps below serve as a guide only. Additional tweaks may be needed for specific experimental conditions or research questions. 

</div>

## Generating a segment size file

`catella` needs to know the name and size of each 'chromosome' or sequence construct. One way to generate a file containing these details is as follows:

```
samtools faidx genome.fa
cut -f1,2 genome.fa.fai > segments.size
```

For the case where we are interested in the nucleosome positions of specific genes in native systems, we generate this file manually based on the region of interest within each gene, treating each one as a 'chromosome'. For example, if we are interested in a 5kbp window around the promoter of three reprogramming factor genes *OCT4*, *SOX2*, and *KLF4*, we can do the following:

```
echo "
OCT4    5000
SOX2    5000
KLF4    5000
" > segments.size
```

## Processing the bam file after alignment

Our goal here is to extract the methylation probabilities of individual bases from the bam file using the package `modkit`. Often this process involves binarizing the signal to discern whether a base is methylated or not, typically done using `modkit call-mods` and setting a filtering threshold on the probability. Instead, `catella` relies on the full probability profile across a molecule to predict nucleosome positions. Hence, one should not perform any filtering on the probability scores before passing the data to `catella`.

<div class="alert alert-info">
    
**Note:** It is important that we preserve the full extent of methylation probability profile during data extraction to increase the sensitivity when predicting nucleosome positions with `catella`.

</div>

### Artificial constructs

For artificial constructs, individual molecules belonging to the same construct typically have the same size covering the full length of the construct or 'chromosome'. As such, we do not need to perform additional steps to standardize the molecules. 

Let us consider the case where the methylation footprinting experiments utilize methyltransferases to modify CpG, GpC, and A. We use the command `modkit extract full` to extract the table of methylation probabilities:

```
modkit extract full --motif A 0 --motif CG 0 --motif GC 1 --mapped --ignore-implicit --ref genome.fa input.bam - | \
awk '$14 != "h"' | bgzip > output.tsv.gz
```

Some points to note about this chain of commands:
* By default, this command outputs both the probabilities of 5-methylcytosine (5mC, or those with `m` in the `mod_code` column of the table) and 5-hydroxymethylcytosine (5hmC, or those with `h`), generating two entries for the same cytosine base. 
Since `catella` only allows one probability score per base, we must resolve this conflict. If we are certain that within the experiments only one form of methylation is done (e.g., 5mC or `m`), we can safely filter out the other form of methylation (e.g., 5hmC or `h`). The filtering `awk '$14 != "h"'` removes any `5hmC` probabilities.
* We explicitly specify the motifs where methylation can occur in the experiments (i.e., `--motif A 0 --motif CG 0 --motif GC 1`) to help remove any background random methylation, particularly on cytosine.
* We use the option `--ignore-implicit` to output modified bases only and ignore canonical (unmodified) ones (those with `-` in the `mod_code` column). This reduces the overall file size of the resulting `tsv` file. In the same spirit, we use the option `--mapped` to only consider molecules that have been successfully aligned.
* We recommend using `bgzip` to compress the `tsv` file to reduce its file size. `catella` supports direct reading of `gzip` file. 

The resulting `output.tsv.gz` file is now ready to be read by `catella`. The specific columns of this table read by `catella` are as follows:

| Column name  | Description                                             |
|--------------|---------------------------------------------------------|
| read_id      | name of the read                                        |
| ref_position | aligned 0-based reference sequence position             |
| chrom        | name of aligned contig                                  |
| ref_strand   | strand of the reference read is aligned to              |
| mod_qual     | probability of the base modifcation in the next column  |
| mod_code     | base modification code from the MM tag                  |

The full description of the output table can be found here - https://nanoporetech.github.io/modkit/intro_extract.html. 

<div class="alert alert-info">
    
**Note:** Remember to keep the header line of the table containing the column names (i.e., do not use the `--no-header` option in `modkit extract full`). `catella` relies on the header to extract data from the relevant columns.

</div>

<div class="alert alert-info">
    
**Note:** `catella` ignores the probability values of canonical bases (i.e., those with `-` in the `mod_code` column). The methylation probability score at these bases is set to zero.

</div>

<div class="alert alert-info">
    
**Note:** Control samples - from our in-house methylation footprinting experiments, we find that the methylation readout based on Oxford Nanopore sequencing tend to be noisier than that from PacBio sequencing. As such, it is useful to also prepare control samples of DNA fibers that are either fully methylated or unmethylated. These can then be used to normalize the signal. See the section Simulation Model which discusses this more in detail.

</div>

### Native fragments

For single-molecule DNA fragments derived from in-vivo systems, they often come in a range of sizes (since they are cut by restriction enzymes) and cover different genomic loci. This is particularly the case for genome-wide methylation footprinting experiments. A typical use case of `catella` in this type of experiments is to predict the nucleosome positions for a selection of loci of interest, such as the promoter regions of some genes.

To this end, we need to standardize the molecule reads and output only the methylation probabilities covering the loci of interest. We first write a `bed` file specifying these loci

```
cat << 'EOF' > genes.bed
chr3	181709500	181714500	SOX2
chr6	31167500	31172500	OCT4
chr9	107487000	107492000	KLF4
EOF
```

Next, we use `bedtools` and this `bed` file to filter the `bam` file, keeping only molecules that fully intersect with the genomic regions we specified:

```
bedtools intersect -abam input.bam -b genes.bed -F 1.0 > intersect.bam
```

Now we can proceed in the same way as above for the artificial constructs:

```
modkit extract full --motif A 0 --motif CG 0 --motif GC 1 --mapped --ignore-implicit --ref genome.fa intersect.bam - | \
awk '$14 != "h"' | bgzip > output.tsv.gz
```
